# 解码策略：从 Logits 到下一个 Token

> 模型做完一次前向计算，只给出一组 **logits**。真正生成文本，还差最后一步：从几万个候选 Token 中选出一个。
>
> 这一章只回答一个问题：**logits 怎么变成 Token？**
>
> 读完后，你应该能直接看懂 Hugging Face、vLLM、SGLang 里的 `temperature`、`top_p`、`top_k`、`repetition_penalty`、`max_tokens`，也知道这些参数改变的是“怎么选”，不是模型参数本身。

上一部分已经讲过模型如何训练。这里不再训练一个玩具 Transformer，而是直接从一组 logits 开始。


In [ ]:
import torch
import torch.nn.functional as F

tokens = ["巴黎", "伦敦", "北京", "东京", "香蕉", "。"]
logits = torch.tensor([4.2, 3.8, 1.2, 0.8, -0.5, 0.4])

probs = F.softmax(logits, dim=-1)
for t, l, p in zip(tokens, logits, probs):
    print(f"{t:>4}  logit={l:>4.1f}  p={p.item():.3f}")


## 1. Greedy：永远选最高分

最直接的方法就是 `argmax`。它稳定、可复现，但不会探索第二选择。

这也是理解后面所有采样方法的起点：**它们都没有重新计算模型，只是在改 logits 到 Token 的选择规则。**


In [ ]:
next_id = torch.argmax(logits).item()
print("Greedy 选择:", tokens[next_id])


## 2. Temperature：先改变分布形状

Temperature 做的事情很简单：

$$
p_i = \mathrm{softmax}(z_i / T)
$$

- `T < 1`：差距被放大，输出更确定
- `T = 1`：保持原分布
- `T > 1`：差距被压平，输出更多样

注意：Temperature **不会删掉候选**，尾部 Token 仍然保留非零概率。


In [ ]:
for T in [0.2, 0.7, 1.0, 1.5]:
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T:<3}: " + ", ".join(f"{t}:{x:.2f}" for t, x in zip(tokens, p.tolist())))


## 3. Top-k / Top-p：把不值得采样的尾部候选砍掉

Temperature 只改变分布形状。真实生成通常还要做截断。

- **Top-k**：只保留最高的 k 个候选
- **Top-p / Nucleus Sampling**：按概率从高到低累加，保留累计概率达到 p 的最小集合

Top-k 控制“最多看几个”，Top-p 控制“至少覆盖多少概率质量”。


In [ ]:
def top_k_filter(logits, k):
    if k is None or k >= logits.numel():
        return logits
    threshold = torch.topk(logits, k).values[-1]
    return torch.where(logits < threshold, torch.tensor(float("-inf")), logits)

def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out[sorted_idx] = sorted_logits
    return out

for name, filtered in [("top_k=2", top_k_filter(logits.clone(), 2)), ("top_p=0.9", top_p_filter(logits.clone(), 0.9))]:
    p = F.softmax(filtered, dim=-1)
    kept = [(t, round(x, 3)) for t, x in zip(tokens, p.tolist()) if x > 0]
    print(name, kept)


## 4. 一次完整采样到底做了什么？

在很多框架里，一次 Decode step 可以粗略理解成：

```text
model forward
    ↓
logits
    ↓ repetition / presence / frequency penalties
temperature
    ↓
top-k / top-p / min-p ...
    ↓
sampling
    ↓
next token
```

不同框架的 processor 顺序可能不同，所以生产环境里要以具体实现为准。


In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None, seed=0):
    torch.manual_seed(seed)
    x = logits.clone() / max(temperature, 1e-5)
    x = top_k_filter(x, top_k)
    if top_p is not None:
        x = top_p_filter(x, top_p)
    probs = F.softmax(x, dim=-1)
    return torch.multinomial(probs, 1).item(), probs

print("greedy:", tokens[torch.argmax(logits).item()])
for name, cfg in [("T=0.7, p=0.9", dict(temperature=0.7, top_p=0.9)), ("T=1.2, p=0.95", dict(temperature=1.2, top_p=0.95))]:
    picks = [tokens[sample_next(logits, seed=s, **cfg)[0]] for s in range(8)]
    print(name, picks)


## 5. 看到厂商参数时，先问它作用在哪一层

| 参数 / 名词 | 作用 |
|---|---|
| `temperature` | 改概率分布形状 |
| `top_k` | 固定候选数量上限 |
| `top_p` | 按累计概率动态截断 |
| `repetition_penalty` | 压低已出现 Token |
| `max_tokens` / `max_new_tokens` | 控制生成长度 |
| `stop` / EOS | 决定何时停止 |
| `seed` | 控制采样随机性 |

下一章不再讨论“选哪个 Token”，而是看更底层的问题：

> **为了得到这组 logits，模型一次推理到底做了什么？为什么慢？**
